<a href="https://colab.research.google.com/github/funviction/MN/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_SVD_KNNBasic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Фінько Вікторія Володимирівна  
**Група:** 3-9  
**Дата:** 13.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідження набору даних
num_users = ratings_df['user_id'].nunique()
num_items = ratings_df['item_id'].nunique()
min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()
missing_values = ratings_df.isnull().sum()

print(f"Кількість унікальних користувачів: {num_users}")
print(f"Кількість унікальних фільмів: {num_items}")
print(f"Мінімальна оцінка: {min_rating}")
print(f"Максимальна оцінка: {max_rating}")
print(f"Середня оцінка: {mean_rating:.4f}")
print("\nРозподіл оцінок:")
print(ratings_df['rating'].value_counts().sort_index())
print("\nПропущені значення по стовпцях:")
print(missing_values)

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.5299

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64

Пропущені значення по стовпцях:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

Набір даних MovieLens 100K містить 100 000 оцінок від 943 унікальних користувачів для 1682 фільмів, при цьому пропущені значення у таблиці повністю відсутні. Оцінки виставлені за цілочисельною шкалою від 1.0 до 5.0 із середнім значенням близько 3.53. Розподіл є зміщеним у бік позитивних оцінок: найчастіше користувачі ставлять 4.0 (понад 34% усіх записів) та 3.0, тоді як оцінка 1.0 зустрічається найрідше.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Розбиття на навчальну та тестову вибірки (80/20)
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print(f"Розмір train_df: {train_df.shape}")
print(f"Розмір test_df:  {test_df.shape}")

Розмір train_df: (80000, 4)
Розмір test_df:  (20000, 4)


In [6]:
# TODO 3.2 — Підготовка об'єктів для бібліотеки Surprise
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

print(f"Кількість записів у train_data: {len(train_data.raw_ratings)}")
print(f"Кількість записів у testset:    {len(testset)}")

Кількість записів у train_data: 80000
Кількість записів у testset:    20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для матричної факторизації SVD обрано три ключові гіперпараметри: кількість прихованих факторів n_factors ([50, 100]) визначає розмірність латентного простору смаків (замале значення призводить до недонавчання, завелике — до перенавчання); швидкість навчання lr_all ([0.005, 0.010]) контролює крок градієнтного спуску; коефіцієнт L2-регуляризації reg_all ([0.02, 0.05]) штрафує великі ваги векторів та запобігає перенавчанню на розрідженій матриці оцінок.

Для колаборативної фільтрації KNNBasic налаштовуються: максимальна кількість сусідів k ([20, 40]), яка задає розмір локального оточення; мінімальний поріг сусідів min_k ([1, 5]), нижче якого прогноз падає до глобального середнього для відсікання шумних оцінок; метрика схожості name (['cosine', 'pearson'] — косинусна схожість та кореляція Пірсона, яка враховує індивідуальний зсув шкали оцінок користувача) та стратегія user_based ([False, True] — порівняння item-based та user-based підходів).

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [9]:
# TODO 5.1 — Підбір гіперпараметрів SVD
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)

In [10]:
# TODO 5.2 — Підбір гіперпараметрів KNNBasic
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)

In [11]:
# TODO 5.3 — Виведення результатів GridSearchCV
svd_cv_rmse = grid_search_svd.best_score['rmse']
svd_best_idx = grid_search_svd.cv_results['rank_test_rmse'].argmin()
svd_cv_mae = grid_search_svd.cv_results['mean_test_mae'][svd_best_idx]
svd_best_params = grid_search_svd.best_params['rmse']

knn_cv_rmse = grid_search_knn.best_score['rmse']
knn_best_idx = grid_search_knn.cv_results['rank_test_rmse'].argmin()
knn_cv_mae = grid_search_knn.cv_results['mean_test_mae'][knn_best_idx]
knn_best_params = grid_search_knn.best_params['rmse']

print("=== Результати SVD (CV) ===")
print(f"Найкращий CV RMSE: {svd_cv_rmse:.4f}")
print(f"Відповідний CV MAE: {svd_cv_mae:.4f}")
print("Найкращі параметри:", svd_best_params)

print("\n=== Результати KNNBasic (CV) ===")
print(f"Найкращий CV RMSE: {knn_cv_rmse:.4f}")
print(f"Відповідний CV MAE: {knn_cv_mae:.4f}")
print("Найкращі параметри:", knn_best_params)

=== Результати SVD (CV) ===
Найкращий CV RMSE: 0.9433
Відповідний CV MAE: 0.7441
Найкращі параметри: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}

=== Результати KNNBasic (CV) ===
Найкращий CV RMSE: 1.0271
Відповідний CV MAE: 0.8143
Найкращі параметри: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
| :--- | :---: | :---: | :--- |
| **SVD** | 0.9433 | 0.7441 | `{'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}` |
| **KNNBasic** | 1.0271 | 0.8143 | `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}` |

На перехресній перевірці модель матричної факторизації **SVD** продемонструвала суттєво вищу точність порівняно з **KNNBasic**: її помилка за метрикою RMSE нижча на 0.0838 (0.9433 проти 1.0271), а за MAE — на 0.0702 (0.7441 проти 0.8143). Це пояснюється тим, що латентне розкладання матриці ефективніше узагальнює приховані вподобання в умовах розріджених даних, тоді як пошук найближчих сусідів чутливіший до нестачі спільних оцінок. Найкращими гіперпараметрами для SVD виявилися `n_factors=100`, швидкість навчання `lr_all=0.01` та регуляризація `reg_all=0.05`, а для KNNBasic — `k=40` сусідів із порогом `min_k=1` та user-based стратегією на основі косинусної подібності (`cosine`).

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [12]:
# TODO 6.1 — Отримання найкращих параметрів за CV RMSE
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# TODO 6.2 — Побудова повної навчальної вибірки trainset
trainset = train_data.build_full_trainset()

# TODO 6.3 — Ініціалізація та навчання фінальних моделей
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [13]:
# TODO 7.1 — Прогнозування на testset
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 — Обчислення метрик якості
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [14]:
# TODO 7.3 — Порівняльна таблиця результатів
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values(by='RMSE', ascending=True).reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919367,0.724672
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
| :--- | :---: | :---: |
| **SVD** | 0.9194 | 0.7247 |
| **KNNBasic** | 1.0120 | 0.7984 |

**Порівняння з CV:** Оцінки помилок на незалежній тестовій вибірці не мають суттєвих розбіжностей із результатами кросвалідації та навіть виявилися дещо нижчими за них: для **SVD** показник RMSE покращився з `0.9433` до `0.9194` (а MAE — з `0.7441` до `0.7247`), а для **KNNBasic** — з `1.0271` до `1.0120` (MAE — з `0.8143` до `0.7984`). Це пояснюється тим, що фінальні моделі навчалися на повній навчальній вибірці (усі 80% даних), тоді як під час 3-фолдової кросвалідації кожен фолд використовував лише дві третини від навчальної частини (~53.3% усіх даних). Співвідношення якості між моделями повністю збереглося, а відсутність зростання помилки на тесті підтверджує стабільність узагальнюючої здатності моделей без перенавчання.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [15]:
user_id = '196'

# TODO 8.1 — Вибір найкращої моделі за CV RMSE
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'

print(f"Обрана модель для рекомендацій: {best_model_name}")

# TODO 8.2 — Перевірка наявності користувача у trainset
try:
    inner_uid = trainset.to_inner_uid(user_id)

    # TODO 8.3 — Отримання списку вже оцінених та неоцінених фільмів
    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}
    all_inner_iids = set(trainset.all_items())
    unrated_inner_iids = all_inner_iids - rated_inner_iids

    # TODO 8.4 — Прогнозування рейтингу для неоцінених фільмів та формування топ-10
    predictions_list = []
    for inner_iid in unrated_inner_iids:
        raw_iid = trainset.to_raw_iid(inner_iid)
        est_rating = best_model.predict(uid=user_id, iid=raw_iid).est
        predictions_list.append((raw_iid, est_rating))

    top_10_df = pd.DataFrame(
        predictions_list,
        columns=['item_id', 'predicted_rating']
    ).sort_values(by='predicted_rating', ascending=False).head(10).reset_index(drop=True)

    print(f"\nТоп-10 рекомендованих фільмів для користувача {user_id}:")
    display(top_10_df)

except ValueError:
    print(f"Помилка: користувач з user_id='{user_id}' відсутній у навчальній вибірці (проблема холодного старту).")

Обрана модель для рекомендацій: SVD

Топ-10 рекомендованих фільмів для користувача 196:


,item_id,predicted_rating
0,318,4.578034
1,427,4.510453
2,603,4.455609
3,1169,4.441489
4,661,4.414566
5,479,4.390852
6,513,4.356216
7,357,4.324538
8,64,4.322364
9,313,4.300572


## 9. Перевірка виконання
Позначте після реалізації:
- [x] Дані ml-100k завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим random_state.
- [x] Reader, train_data і testset створені коректно.
- [x] GridSearchCV для обох моделей виконано лише на train_data.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано RMSE та MAE для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

---

## 10. Висновки

1. **Обрана модель за кросвалідацією:** Для побудови рекомендацій обрано модель **SVD**, оскільки на 3-фолдовій перехресній перевірці вона показала суттєво нижчу помилку прогнозування (`CV RMSE = 0.9433`, `CV MAE = 0.7441`) порівняно з `KNNBasic` (`CV RMSE = 1.0271`, `CV MAE = 0.8143`).
2. **Найкращі гіперпараметри:** Для **SVD** оптимальною виявилася конфігурація `{'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}`, а для **KNNBasic** — `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}`.
3. **Підсумкові значення на тестовій вибірці:** На незалежній тестовій вибірці **SVD** досягла `Test RMSE = 0.9194` та `Test MAE = 0.7247`, тоді як **KNNBasic** отримала `Test RMSE = 1.0120` та `Test MAE = 0.7984`.
4. **Різниця між CV та тестуванням:** Результати на тестовій вибірці узгоджуються з кросвалідацією та є дещо кращими (RMSE знизився на `0.0239` для SVD і на `0.0151` для KNNBasic), оскільки фінальні моделі навчалися на повних 80% навчальних даних, а не на двох третинах вибірки під час кросвалідації.
5. **Обмеження отриманих рекомендацій:** Обидва алгоритми належать до чистої колаборативної фільтрації та страждають від проблеми «холодного старту» — для нових користувачів або нових фільмів без історії оцінок модель не може сформувати персоналізований прогноз і повертає глобальний середній рейтинг.

**Висновок:** У ході лабораторної роботи на наборі даних MovieLens 100K (100 000 оцінок від 943 користувачів для 1682 фільмів) було реалізовано та порівняно моделі матричної факторизації `SVD` і колаборативної фільтрації `KNNBasic`. За результатами кросвалідації модель `SVD` з параметрами `n_factors=100`, `lr_all=0.01` та `reg_all=0.05` перевершила `KNNBasic` (`k=40`, `min_k=1`, `cosine`, `user_based=True`), продемонструвавши CV RMSE `0.9433` проти `1.0271`. На незалежній тестовій вибірці перевага `SVD` підтвердилася: підсумкові метрики склали `RMSE = 0.9194` та `MAE = 0.7247`, тоді як для `KNNBasic` вони становили `RMSE = 1.0120` та `MAE = 0.7984`. Незначне покращення тестових показників порівняно з кросвалідацією пояснюється збільшенням обсягу навчальної вибірки при фінальному навчанні та підтверджує відсутність перенавчання. На основі моделі `SVD` було успішно згенеровано топ-10 персоналізованих рекомендацій для користувача `'196'` із виключенням уже переглянутих ним фільмів. Основним обмеженням побудованої системи залишається неможливість персоналізації для нових користувачів та об'єктів без історії взаємодій (проблема холодного старту), що в реальних проєктах вирішується додаванням контентних ознак.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [16]:
class CustomBiasedSVD:
    """
    Власна реалізація алгоритму рекомендацій на базі матричної факторизації
    зі зміщеннями (Biased SVD) та оптимізацією методом SGD.
    """
    def __init__(self, n_factors=40, lr=0.007, reg=0.05, n_epochs=20, random_state=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.random_state = random_state

    def fit(self, df):
        rng = np.random.default_rng(self.random_state)

        # Створення мапінгу зовнішніх ID у внутрішні індекси
        unique_users = df['user_id'].unique()
        unique_items = df['item_id'].unique()

        self.user_to_idx = {u: idx for idx, u in enumerate(unique_users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(unique_items)}
        self.idx_to_item = {idx: i for i, idx in self.item_to_idx.items()}

        n_users = len(unique_users)
        n_items = len(unique_items)

        # Глобальне середнє та вектори зміщень
        self.global_mean = df['rating'].mean()
        self.bu = np.zeros(n_users, dtype=np.float64)
        self.bi = np.zeros(n_items, dtype=np.float64)

        # Матриці латентних факторів користувачів (P) та об'єктів (Q)
        self.P = rng.normal(0, 0.1, size=(n_users, self.n_factors))
        self.Q = rng.normal(0, 0.1, size=(n_items, self.n_factors))

        # Перетворення навчальних даних у масиви індексів
        u_indices = df['user_id'].map(self.user_to_idx).to_numpy()
        i_indices = df['item_id'].map(self.item_to_idx).to_numpy()
        ratings = df['rating'].to_numpy(dtype=np.float64)

        # Запам'ятовуємо оцінені фільми для кожного користувача
        self.user_rated_items = {}
        for u, i in zip(unique_users, unique_items):
            pass
        for u_id, group in df.groupby('user_id'):
            self.user_rated_items[u_id] = set(group['item_id'])

        n_samples = len(ratings)

        # Стохастичний градієнтний спуск по епохах
        for epoch in range(self.n_epochs):
            order = rng.permutation(n_samples)
            for idx in order:
                u = u_indices[idx]
                i = i_indices[idx]
                r = ratings[idx]

                # Прогноз та помилка
                pred = self.global_mean + self.bu[u] + self.bi[i] + np.dot(self.P[u], self.Q[i])
                err = r - pred

                # Оновлення зміщень
                self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                self.bi[i] += self.lr * (err - self.reg * self.bi[i])

                # Оновлення латентних векторів
                p_u_old = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * p_u_old - self.reg * self.Q[i])

        return self

    def predict_one(self, user_id, item_id):
        u_known = user_id in self.user_to_idx
        i_known = item_id in self.item_to_idx

        pred = self.global_mean
        if u_known:
            pred += self.bu[self.user_to_idx[user_id]]
        if i_known:
            pred += self.bi[self.item_to_idx[item_id]]
        if u_known and i_known:
            u = self.user_to_idx[user_id]
            i = self.item_to_idx[item_id]
            pred += np.dot(self.P[u], self.Q[i])

        return float(np.clip(pred, 1.0, 5.0))

    def evaluate(self, test_dataframe):
        preds = np.array([
            self.predict_one(u, i)
            for u, i in zip(test_dataframe['user_id'], test_dataframe['item_id'])
        ])
        actuals = test_dataframe['rating'].to_numpy(dtype=np.float64)
        rmse = np.sqrt(np.mean((actuals - preds) ** 2))
        mae = np.mean(np.abs(actuals - preds))
        return rmse, mae

    def recommend_top_n(self, user_id, n=10):
        if user_id not in self.user_to_idx:
            raise ValueError(f"Користувач {user_id} відсутній у навчальній вибірці.")

        rated = self.user_rated_items.get(user_id, set())
        u_idx = self.user_to_idx[user_id]

        # Векторизований підрахунок рейтингів для всіх фільмів одразу
        all_scores = self.global_mean + self.bu[u_idx] + self.bi + self.Q @ self.P[u_idx]
        all_scores = np.clip(all_scores, 1.0, 5.0)

        candidates = [
            (self.idx_to_item[i_idx], float(all_scores[i_idx]))
            for i_idx in range(len(self.idx_to_item))
            if self.idx_to_item[i_idx] not in rated
        ]

        candidates.sort(key=lambda x: x[1], reverse=True)
        return pd.DataFrame(candidates[:n], columns=['item_id', 'predicted_rating'])


# Тестування власного алгоритму на тих самих train_df та test_df
custom_model = CustomBiasedSVD(n_factors=50, lr=0.007, reg=0.05, n_epochs=25, random_state=42)
custom_model.fit(train_df)

custom_rmse, custom_mae = custom_model.evaluate(test_df)
print(f"CustomBiasedSVD — Test RMSE: {custom_rmse:.4f}, Test MAE: {custom_mae:.4f}")

print("\nТоп-10 рекомендацій від власного алгоритму для користувача '196':")
display(custom_model.recommend_top_n('196', n=10))

CustomBiasedSVD — Test RMSE: 0.9207, Test MAE: 0.7256

Топ-10 рекомендацій від власного алгоритму для користувача '196':


,item_id,predicted_rating
0,64,4.455595
1,318,4.445566
2,1194,4.414555
3,114,4.406867
4,603,4.377271
5,483,4.373550
6,480,4.359198
7,513,4.323800
8,170,4.320322
9,98,4.315660
